#### This sript is a standalone test script (not part of the final application)
#### for training an XGBoost model using preprocessed data
Goal is to test the change in prediction accuracy by using 
## 1. different hyperparameters 
## 2. additional input parameters and 
## 3. additional training data

In [4]:
# FROST fetch: server-side filtering + compact payload + parallel requests
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- Config: your datastreams + date window (UTC) ---
datastreams = [
    ("BG.West.010", "https://multicare.bk.tudelft.nl/FROST-Server/v1.0/Datastreams(1)/Observations?$orderby=phenomenonTime desc"),
    ("BG.West.270", "https://multicare.bk.tudelft.nl/FROST-Server/v1.0/Datastreams(7)/Observations?$orderby=phenomenonTime desc"),
    ("01.West.120", "https://multicare.bk.tudelft.nl/FROST-Server/v1.0/Datastreams(13)/Observations?$orderby=phenomenonTime desc"),
]
start_date = pd.Timestamp("2025-01-01", tz="UTC")
end_date   = pd.Timestamp("2025-05-31 23:59:59", tz="UTC")

# --- Tuning knobs ---
PAGE_SIZE   = 1000   # try larger if the server allows (e.g., 2000)
TIMEOUT     = 30     # per-request timeout (seconds)
MAX_WORKERS = 3      # one per datastream

# --- Helper: phenomenonTime can be 'start/end' or a single instant ---
def parse_phenomenon_time(value: str) -> pd.Timestamp:
    # Most FROST servers return instants; keep split for safety if an interval appears
    if isinstance(value, str) and "/" in value:
        value = value.split("/")[0]
    return pd.to_datetime(value, errors="coerce", utc=True)

# --- Build a filtered, compact URL (drops $orderby, adds $filter/$select/$top) ---
def build_filtered_url(base_url: str, start: pd.Timestamp, end: pd.Timestamp, top: int) -> str:
    # Remove any $orderby to avoid expensive server-side sorting
    if "$orderby=" in base_url:
        parts = [p for p in base_url.split("&") if not p.startswith("$orderby=")]
        base_url = "&".join(parts)

    sep = "&" if "?" in base_url else "?"
    start_iso = start.strftime("%Y-%m-%dT%H:%M:%SZ")
    end_iso   = end.strftime("%Y-%m-%dT%H:%M:%SZ")

    additions = [
        f"$filter=phenomenonTime ge {start_iso} and phenomenonTime le {end_iso}",
        "$select=phenomenonTime,result",
        f"$top={top}",
        # If your server supports it, uncomment the next line for even faster/leaner responses:
        # "$resultFormat=dataArray"
    ]
    return base_url + sep + "&".join(additions)

# --- Core fetch: follow @iot.nextLink only (no extra request per page) ---
def fetch_all_filtered(base_url: str, start: pd.Timestamp, end: pd.Timestamp, page_size: int = 1000, session: requests.Session | None = None) -> pd.DataFrame:
    sess = session or requests.Session()
    headers = {"Accept": "application/json;odata.metadata=none", "Accept-Encoding": "gzip, deflate"}

    url = build_filtered_url(base_url, start, end, page_size)
    rows = []

    while url:
        r = sess.get(url, headers=headers, timeout=TIMEOUT)
        r.raise_for_status()
        payload = r.json()

        vals = payload.get("value", [])
        if not vals:
            break

        rows.extend(
            {"timestamp": parse_phenomenon_time(o.get("phenomenonTime")), "internal_temp": o.get("result")}
            for o in vals
        )

        url = payload.get("@iot.nextLink")  # follow server paging

    if not rows:
        return pd.DataFrame(columns=["timestamp", "internal_temp"])

    df = pd.DataFrame(rows)
    # Ensure types + clean
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
    df["internal_temp"] = pd.to_numeric(df["internal_temp"], errors="coerce")
    df = (
        df.dropna(subset=["timestamp"])
          .drop_duplicates(subset=["timestamp", "internal_temp"])  # dedupe by both fields
          .sort_values("timestamp")
          .reset_index(drop=True)
    )
    return df

# --- Wrapper to fetch one stream (with safety) ---
def fetch_one(name_url):
    name, url = name_url
    print(f"Fetching {name} ...")
    try:
        with requests.Session() as sess:
            df = fetch_all_filtered(url, start_date, end_date, page_size=PAGE_SIZE, session=sess)
        print(f"  {name}: {len(df)} rows")
        return name, df
    except Exception as e:
        print(f"  {name}: ERROR -> {e}")
        return name, pd.DataFrame(columns=["timestamp", "internal_temp"])

# --- Parallel fetching for all streams ---
sensor_data: dict[str, pd.DataFrame] = {}
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(fetch_one, pair): pair[0] for pair in datastreams}
    for fut in as_completed(futures):
        name, df = fut.result()
        sensor_data[name] = df

# --- Combine for analysis/plotting (adds sensor label) ---
combined = (
    pd.concat([df.assign(sensor=name) for name, df in sensor_data.items()], ignore_index=True)
    if sensor_data else
    pd.DataFrame(columns=["timestamp", "internal_temp", "sensor"])
)

print("\nCombined head():")
print(combined.head())
print("\nCounts by sensor:")
print(combined.groupby("sensor")["internal_temp"].count())

# --- Hourly average ---
hourly_avg = (
    combined
    .set_index("timestamp")              # make timestamp the index
    .groupby("sensor")                   # keep averages separate per sensor
    .resample("1h")                       # hourly bins
    .mean(numeric_only=True)              # mean of numeric columns
    .reset_index()
)

print("\nHourly average head():")
print(hourly_avg.head())

Fetching BG.West.010 ...
Fetching BG.West.270 ...
Fetching 01.West.120 ...
  BG.West.270: 8362 rows  BG.West.010: 8362 rows

  01.West.120: 7731 rows

Combined head():
                  timestamp  internal_temp       sensor
0 2025-03-23 12:14:39+00:00           22.9  BG.West.010
1 2025-03-23 12:24:44+00:00           23.0  BG.West.010
2 2025-03-23 12:34:49+00:00           23.0  BG.West.010
3 2025-03-23 12:44:55+00:00           23.1  BG.West.010
4 2025-03-23 12:55:00+00:00           23.0  BG.West.010

Counts by sensor:
sensor
01.West.120    7731
BG.West.010    8362
BG.West.270    8362
Name: internal_temp, dtype: int64

Hourly average head():
        sensor                 timestamp  internal_temp
0  01.West.120 2025-03-23 12:00:00+00:00      23.275000
1  01.West.120 2025-03-23 13:00:00+00:00      22.950000
2  01.West.120 2025-03-23 14:00:00+00:00      22.800000
3  01.West.120 2025-03-23 15:00:00+00:00      22.633333
4  01.West.120 2025-03-23 16:00:00+00:00      22.516667


Now that data from sensors is in a readable dataframe, I want to combine this with the external sensor data from NetCDF file (downloaded from Davis weather station data at Green Village)

In [6]:
# === External weather from NetCDF (Jan–May 2025) ===
import pandas as pd
import xarray as xr
from pathlib import Path
#  path for the NetCDF file downloaded from Davis weather station data at Green Village
nc_files = [
    "../input/ExternalTemp/davis-TUD-GV_Green_Village_202501.nc",
    "../input/ExternalTemp/davis-TUD-GV_Green_Village_202502.nc",
    "../input/ExternalTemp/davis-TUD-GV_Green_Village_202503.nc",
    "../input/ExternalTemp/davis-TUD-GV_Green_Village_202504.nc",
    "../input/ExternalTemp/davis-TUD-GV_Green_Village_202505.nc",
]

# 2) Helper to read one NetCDF into a tidy DataFrame
def _read_nc_to_df(path: str | Path) -> pd.DataFrame:
    ds = xr.open_dataset(path)
    df = ds.to_dataframe().reset_index()

    # Keep only relevant columns if present
    keep_time_cols = [c for c in ["epoch_time", "time", "time_as_string"] if c in df.columns]
    keep_sensor_cols = [c for c in ["pressure", "temperature", "humidity", "solar_irradiance", "wind_speed"] if c in df.columns]

    return df[keep_time_cols + keep_sensor_cols]
# 3) Read and combine all months
ext_raw_list = []
for f in nc_files:
    try:
        ext_raw_list.append(_read_nc_to_df(f))
    except Exception as e:
        print(f"[WARN] Failed reading {f}: {e}")

if ext_raw_list:
    ext_raw = pd.concat(ext_raw_list, ignore_index=True)
else:
    ext_raw = pd.DataFrame(columns=["epoch_time", "pressure", "temperature", "humidity", "solar_irradiance", "wind_speed"])

# 4) Coerce selected sensor columns to numeric
sensor_cols = [c for c in ["pressure", "temperature", "humidity", "solar_irradiance", "wind_speed"] if c in ext_raw.columns]
for c in sensor_cols:
    ext_raw[c] = pd.to_numeric(ext_raw[c], errors="coerce")

# 5) Convert epoch_time → UTC timestamp (to match FROST timestamps)
ext_raw["timestamp_utc"] = pd.to_datetime(pd.to_numeric(ext_raw["epoch_time"], errors="coerce"), unit="s", utc=True)

# 6) Basic cleanup: drop bad timestamps, sort, and drop exact duplicates
ext_clean = (
    ext_raw.dropna(subset=["timestamp_utc"])
           .drop_duplicates(subset=["timestamp_utc"] + sensor_cols)  # dedupe on time+values
           .sort_values("timestamp_utc")
           .reset_index(drop=True)
)

# 7) Index by time and resample to hourly means
ext_hourly = (
    ext_clean.set_index("timestamp_utc")[sensor_cols]  # index on UTC time; keeping only sensor cols
             .resample("1h")
             .mean()                                   # NaNs ignored; all-NaN hour remains NaN
             .reset_index()
)

# ---- local time (e.g., Europe/Amsterdam) ----
ext_hourly["time_local"] = ext_hourly["timestamp_utc"].dt.tz_convert("Europe/Amsterdam")

#checks
print("External hourly head():")
print(ext_hourly.head())
print(f"Total rows in ext_hourly: {len(ext_hourly)}")
print("Counts per column (non-NaN):")
print(ext_hourly[sensor_cols].count())


External hourly head():
              timestamp_utc     pressure  temperature  humidity  \
0 2025-01-01 00:00:00+00:00  1015.264587          NaN       NaN   
1 2025-01-01 01:00:00+00:00  1014.984619          NaN       NaN   
2 2025-01-01 02:00:00+00:00  1014.504272          NaN       NaN   
3 2025-01-01 03:00:00+00:00  1014.489624          NaN       NaN   
4 2025-01-01 04:00:00+00:00  1014.516235          NaN       NaN   

   solar_irradiance  wind_speed                time_local  
0               NaN         NaN 2025-01-01 01:00:00+01:00  
1               NaN         NaN 2025-01-01 02:00:00+01:00  
2               NaN         NaN 2025-01-01 03:00:00+01:00  
3               NaN         NaN 2025-01-01 04:00:00+01:00  
4               NaN         NaN 2025-01-01 05:00:00+01:00  
Total rows in ext_hourly: 3625
Counts per column (non-NaN):
pressure            3566
temperature         3289
humidity            3331
solar_irradiance    3331
wind_speed          3328
dtype: int64
